# Crawling Berita

In [ ]:
!pip install requests
!pip install beautifulsoup4
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
!pip install builtwith

  Preparing metadata (setup.py) ... done
  Created wheel for builtwith: filename=builtwith-1.3.4-py3-none-any.whl size=36077 sha256=bbf93d04d625180933c22fd6c25e231a71f66995b5daff32fdacba6907a3addc
  Stored in directory: /root/.cache/pip/wheels/7f/2d/b2/606e3df914d4aeeab99c4a4e3e9a61673d2293c2e346db00c8
Successfully built builtwith


**Berbagai Berita**

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def scrape_kompas_article(url):
    """
    Mengambil judul, isi, dan kategori berita dari sebuah URL Kompas.com.

    Args:
        url (str): URL dari artikel berita Kompas.com.
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        r = requests.get(url, headers=headers)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error saat mengambil URL: {e}")
        return

    soup = BeautifulSoup(r.content, "html.parser")

    judul = soup.select_one("h1.read__title")
    judul_text = judul.text.strip() if judul else "Tidak ditemukan judul"

    isi_elems = soup.select("div.read__content p")
    isi_text = "\n".join([p.get_text(strip=True) for p in isi_elems]) if isi_elems else "Tidak ditemukan isi"

    kategori_meta = soup.find("meta", {"property": "article:section"})
    if kategori_meta and kategori_meta.get("content"):
        kategori_text = kategori_meta["content"].strip()
    else:
        try:
            kategori_map = {
                "edu": "Edukasi",
                "health": "Kesehatan",
                "money": "Ekonomi",
                "lifestyle": "Lifestyle",
                "nasional": "Nasional",
                "tekno": "Teknologi",
                "metro": "Metro",
                "megapolitan": "Megapolitan",
                "travel": "Travel"
            }

            domain_parts = url.split("//")[1].split("/")[0].split(".")
            kategori_text = "Tidak ditemukan kategori"

            if len(domain_parts) > 2 and domain_parts[0] in kategori_map:
                kategori_text = kategori_map[domain_parts[0]]
            else:
                segmen_parts = url.split("kompas.com/")[1].split("/")
                for part in segmen_parts:
                    if part in kategori_map:
                        kategori_text = kategori_map[part]
                        break
        except (IndexError, AttributeError):
            kategori_text = "Tidak ditemukan kategori"

    print("--------------------------------------------------")
    print("Judul Berita  :", judul_text)
    print("Kategori      :", kategori_text)
    print("--- Isi Berita ---")
    print(isi_text)
    print("--------------------------------------------------")

urls_to_test = [
    "https://lifestyle.kompas.com/read/2025/08/31/100000720/waspadai-kelelahan-mental-akibat-kebanyakan-berita-negatif-",
    "https://money.kompas.com/read/2025/08/18/134653726/pendidikan-kewirausahaan-yang-merdeka",
    "https://health.kompas.com/read/25H19130000068/dokter--olahraga-bisa-turunkan-risiko-kanker-asal-rutin-dan-benar",
    "https://nasional.kompas.com/read/2025/09/03/18022971/peristiwa-gas-air-mata-unisba-mendikti-janjikan-pendampingan-dan?utm_source=Various&utm_medium=Referral&utm_campaign=Top_Desktop",
    "https://nasional.kompas.com/read/2025/08/19/07251961/harapan-dan-catatan-soal-anggaran-pendidikan-terbesar-sepanjang-sejarah-ri?utm_source=Various&utm_medium=Referral&utm_campaign=Top_Desktop",
    "https://travel.kompas.com/read/2025/09/04/210100827/berdarah-belanda-depok-pesepak-bola-miliano-jonathans-resmi-jadi-wni?utm_source=Various&utm_medium=Referral&utm_campaign=Top_Desktop"
]

for test_url in urls_to_test:
    scrape_kompas_article(test_url)

--------------------------------------------------
Judul Berita  : Waspadai Kelelahan Mental akibat Kebanyakan Berita Negatif
Kategori      : Lifestyle
--- Isi Berita ---

KOMPAS.com -Berbagai informasi peristiwa terbaru sekarang bisa kita dapatkan tanpa henti, baik melalui media arus utama atau media sosial.
Tanpa disadari kondisi itu berpengaruh besar pada kondisi mental seseorang. Para ahli menyebut fenomena ini sebagai "headline stress disorder" untuk menggambarkan kondisi stres akibat terlalu sering terpapar berita bernuansa negatif.
Psikolog sosial Dicky Pelupessy Ph.D menyebutkan, emosi sedih, gelisah, atau marah yang timbul tersebut sebenarnya hal yang normal.
"Ini adalah perasaan yang normal, karena peristiwa politik, apalagi yang luar biasa, tidak terjadi setiap hari, sehingga akan membangkitkan emosi negatif," katanya ketika dihubungi Kompas.com (29/8/2025).
Baca juga:Mengapa Bisa Cemas Setelah Lihat Berita tentang Kondisi Negara? Ini Kata Psikolog
Pengajar dari Fakultas Psi

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urlparse

def scrape_kompas_article(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        r = requests.get(url, headers=headers)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error saat mengambil URL: {e}")
        return None

    soup = BeautifulSoup(r.content, "html.parser")

    # Judul
    judul = soup.select_one("h1.read__title")
    judul_text = judul.text.strip() if judul else "Tidak ditemukan judul"

    # Isi (100 kata)
    isi_elems = soup.select("div.read__content p")
    isi_text = " ".join([p.get_text(strip=True) for p in isi_elems]) if isi_elems else "Tidak ditemukan isi"
    words = isi_text.split()
    isi_100 = " ".join(words[:100]) + ("..." if len(words) > 100 else "")

    # Cari kategori
    kategori_text = "Tidak ditemukan kategori"

    kategori_meta = soup.find("meta", {"property": "article:section"})
    if kategori_meta and kategori_meta.get("content"):
        kategori_text = kategori_meta["content"].strip()

    if kategori_text == "Tidak ditemukan kategori":
        breadcrumb = soup.select("div.breadcrumb__link a")
        if breadcrumb and len(breadcrumb) > 1:
            kategori_text = breadcrumb[1].get_text(strip=True)

    if kategori_text == "Tidak ditemukan kategori":
        parsed = urlparse(url)
        subdomain = parsed.netloc.split(".")[0]
        if subdomain and subdomain != "www" and subdomain != "kompas":
            kategori_text = subdomain.capitalize()

    return {
        "Judul": judul_text,
        "Isi (100 kata)": isi_100,
        "Kategori": kategori_text
    }

# Daftar URL
urls_to_test = [
    "https://lifestyle.kompas.com/read/2025/08/31/100000720/waspadai-kelelahan-mental-akibat-kebanyakan-berita-negatif-",
    "https://money.kompas.com/read/2025/08/18/134653726/pendidikan-kewirausahaan-yang-merdeka",
    "https://health.kompas.com/read/25H19130000068/dokter--olahraga-bisa-turunkan-risiko-kanker-asal-rutin-dan-benar",
    "https://nasional.kompas.com/read/2025/09/03/18022971/peristiwa-gas-air-mata-unisba-mendikti-janjikan-pendampingan-dan",
    "https://nasional.kompas.com/read/2025/08/19/07251961/harapan-dan-catatan-soal-anggaran-pendidikan-terbesar-sepanjang-sejarah-ri",
    "https://travel.kompas.com/read/2025/09/04/210100827/berdarah-belanda-depok-pesepak-bola-miliano-jonathans-resmi-jadi-wni"
]

results = [scrape_kompas_article(u) for u in urls_to_test if scrape_kompas_article(u)]
df = pd.DataFrame(results)

from IPython.display import display
display(df)

,Judul,Isi (100 kata),Kategori
0,Waspadai Kelelahan Mental akibat Kebanyakan Be...,KOMPAS.com -Berbagai informasi peristiwa terba...,Lifestyle
1,Pendidikan Kewirausahaan yang Merdeka,"SETIAP17 Agustus, Masyarakat Indonesia merayak...",Money
2,"Dokter: Olahraga Bisa Turunkan Risiko Kanker, ...",KOMPAS.com –Health Management Specialist Corpo...,Health
3,"Peristiwa Gas Air Mata Unisba, Mendikti Janjik...","JAKARTA, KOMPAS.com- Menteri Pendidikan Tinggi...",Nasional
4,Harapan dan Catatan soal Anggaran Pendidikan T...,"JAKARTA, KOMPAS.com- Presiden Prabowo Subianto...",Nasional
5,"Berdarah Belanda Depok, Pesepak Bola Miliano J...",KOMPAS.com -Kementerian Hukum Republik Indones...,Travel
